# Fase 7 · Comparación de modelos de clasificación

Esta fase responde al objetivo específico 7: hasta dónde permite la información del
certificado de nacido vivo identificar un nacimiento en riesgo de bajo peso, y cuánto añaden
los determinantes sociales a esa identificación por encima de la información obstétrica.

Se comparan los **siete modelos** descritos en la Sección 3.8 del marco teórico: regresión
logística, sus versiones con penalización lasso y ridge, árbol de decisión, bosque aleatorio,
*gradient boosting* y perceptrón multicapa.

Además se ejecuta un **contraste de especificación** sobre el *gradient boosting*: el mismo
estimador, con los mismos hiperparámetros y sobre el mismo pliegue, cambiando únicamente si
las covariables se declaran categóricas. No es un octavo modelo, y las tablas generales no lo
incluyen.

## Por qué este cuaderno llama a un guion en lugar de contener el análisis

El procedimiento vive en `fase7_portatil.py`, en la raíz del proyecto, y aquí solo se invoca.
La razón es que esta fase debe poder ejecutarse en un equipo distinto del que alberga el
proyecto, por su coste de cómputo. El guion está escrito para viajar solo: su paso
`preparar` deja una carpeta autosuficiente que se entrena en otra máquina sin necesidad del
resto del código.

Mantener aquí una segunda copia del procedimiento fue lo que en su momento permitió que el
manuscrito describiera una salida que el cuaderno no producía. Con esta disposición hay una
sola implementación y dos formas de lanzarla.

## Preparación

Se localiza la raíz del proyecto subiendo desde el directorio de trabajo hasta encontrar
`src/config.py`, que es la marca de la raíz, y se importa el guion como un módulo más. Esto
funciona con independencia de dónde esté instalada la carpeta del proyecto.

In [ ]:
import sys
import importlib
from pathlib import Path

# La raiz del proyecto es la primera carpeta, subiendo, que contiene src/config.py.
# No se escribe una ruta fija: asi el cuaderno funciona en cualquier equipo.
RAIZ = next(c for c in [Path.cwd(), *Path.cwd().parents]
            if (c / "src" / "config.py").exists())
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "src"))
# Los guiones que se ejecutan por separado viven en guiones/. fase7_portatil es
# uno de ellos: este cuaderno lo importa en lugar de repetir su codigo.
sys.path.insert(0, str(RAIZ / "guiones"))

# Se importa el guion como modulo. Sus funciones preparar() y entrenar() son las
# mismas que se ejecutan desde la terminal; no hay una segunda version del codigo.
import fase7_portatil as fase7
importlib.reload(fase7)

print("raiz del proyecto :", RAIZ)
print("guion             :", Path(fase7.__file__).name)
print("clave de agrupacion:", fase7.CLAVE_GRUPO)

La salida confirma tres cosas: que la raíz se localizó, que el guion se importó, y cuál
es la clave de agrupación que viaja con los datos. Esa clave es el municipio de residencia, y
**no es un predictor**: se emplea únicamente para que el remuestreo bootstrap y la partición
espacial respeten la estructura jerárquica de la muestra.

## Paso 1. Preparar los datos

Se leen la muestra armonizada que dejó `01b_armonizacion` y la tabla de cribado que dejó
`02_cribado_mi`, se conservan únicamente las variables retenidas por el cribado, y el
resultado se escribe en una carpeta `envio_fase7/`.

Ese recorte es lo que hace viable entrenar en otra máquina: en lugar de los 1,25 GB de la
matriz completa viaja solo la submuestra con las columnas necesarias.

In [ ]:
fase7.preparar()

La salida enumera qué quedó en `envio_fase7/`: el parquet con los datos recortados, la
lista de predictores y una copia del propio guion. Si el paso se detiene indicando que no
encuentra un archivo de origen, falta ejecutar antes `01b_armonizacion` o `02_cribado_mi`.

## Paso 2. Entrenar y evaluar

El guion busca los datos en su propia carpeta, porque está pensado para ejecutarse junto al
parquet en el equipo de entrenamiento. Al lanzarlo desde aquí, ambos pasos ocurren en la
misma máquina, de modo que se le indica dónde quedaron los datos y dónde escribir las
salidas.

Esta celda es la más costosa de todo el trabajo: ajusta los siete modelos sobre dos
esquemas de partición, calcula la importancia por permutación y los intervalos por
remuestreo.

In [ ]:
# El guion resuelve sus rutas respecto de su propia ubicacion, porque esta
# disenado para viajar junto al parquet. Al ejecutarlo aqui, los dos pasos
# ocurren en el mismo equipo, de modo que se le indican las rutas que dejo
# preparar(). Se modifican las variables del modulo, no el archivo.
fase7.DATOS = fase7.DIR_ENVIO / "datos_fase7.parquet"
fase7.SALIDAS = RAIZ / "salidas_fase7"

print("datos  :", fase7.DATOS)
print("salidas:", fase7.SALIDAS)
print()

fase7.entrenar()

## Lectura de las salidas

Las tablas quedan en `salidas_fase7/`, en `.csv` para revisar y en `.tex` para insertar en el
manuscrito. Cada una corresponde a una subsección de la Sección 5.6:

| Archivo | Va a |
|---|---|
| `clasificacion_particion_aleatoria` | Desempeño en el conjunto de prueba |
| `clasificacion_aporte_social` | Cuánto añaden los determinantes sociales |
| `clasificacion_comparacion_particiones` | Desempeño bajo partición temporal |
| `clasificacion_importancia` | Importancia de variables por modelo |
| `clasificacion_matrices_confusion` | Matrices de confusión |
| `clasificacion_intervalos` | Intervalos bootstrap y modelo seleccionado |
| `clasificacion_especificacion_categorica` | Contraste de la especificación categórica |

**Las tablas generales contienen siete filas.** El contraste de especificación aparece
únicamente en su propio cuadro, con dos filas, porque no es un modelo adicional sino dos
especificaciones del séptimo.

La cifra de referencia para leer cualquier AUC-PR es la **línea base**, que equivale a la
prevalencia del desenlace en el conjunto de prueba: un modelo que no aporte información se
sitúa en ese valor. La ganancia sobre esa línea, y no el valor absoluto, es lo que mide la
capacidad de identificación.

De esta fase **no se sigue ninguna afirmación causal**. Un modelo de clasificación ordena
observaciones según su probabilidad estimada; el efecto de cada determinante lo estiman los
modelos multinivel de la Sección 5.4.